# SKRIPSI: Train YOLOv12

---
*   Dataset: [Dataset Skripsi_V1](https://app.roboflow.com/skripsian-bwueb/skripsi_v1/2/)


## Overview notebook <span style="color:cyan">VERSION 2</span> (26 April 2026):  
*   **`Model`**: <span style="color:yellow">**YOLOv12-small**</span> 1024px
*   **`Dataset`**: Stratified split by monitor type 70:15:15 + OOD monitor type test
*   **`OOD test`**: `type-009`, `type-011`, `type-021`. Represents layout monitor kiri, kanan, tengah
*   V1 failed karena belum add secret jir
*   V2 failed Out of Memory karena batch size kegedean

## **STEP 1: Environment and Library Setup**

### 1.1 - Configure API keys
- Go to your [`Roboflow Settings`](https://app.roboflow.com/settings/api) page. Click `Copy`. This will place your private key in the clipboard.
- In Kaggle, go to the `Add-ons` and click on `Secrets` (🔑). Store Roboflow API Key under the name `ROBOFLOW_API_KEY`.

### 1.2 - Configure GPU

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Settings` -> `Accelerator`, set it to `GPU T4x2`, and then click `Save`.

In [1]:
!nvidia-smi
!python --version
!pip show torch

Sun Apr 26 17:34:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 1.3 - Create a `HOME` constant.

In [2]:
import os
HOME = os.getcwd()
print(HOME)

/kaggle/working


### 1.4 - Install Dependencies

In [3]:
!pip install packaging ninja
!ninja --version
!echo $?

1.13.0.git.kitware.jobserver-pipe-1
0


In [4]:
!pip install -q git+https://github.com/sunsmarterjie/yolov12.git roboflow supervision wandb
!pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl --no-build-isolation

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 111.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but y

In [5]:
import ultralytics
ultralytics.checks()

Ultralytics 8.3.63 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6843.3/8062.4 GB disk)


In [6]:
#IMPORT EVERYTHING HERE
# Yolo
from ultralytics import YOLO

# Setup wandb for monitoring
import wandb
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
ROBOFLOW_API_KEY = user_secrets.get_secret("ROBOFLOW_API_KEY")
WANDB_API_KEY = user_secrets.get_secret("WANDB_API_KEY")

# For directory
import os

## **STEP 2: Download Dataset via Code from Roboflow**

### 2.1 - Get datasets from Roboflow

In [7]:
os.chdir("/kaggle/working/")

print("DOWNLOADING DATASET FROM ROBOFLOW")
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("skripsian-bwueb").project("skripsi_v1")
version = project.version(4)
dataset = version.download("yolov12")

DOWNLOADING DATASET FROM ROBOFLOW
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Skripsi_V1-4 in yolov12:: 100%|██████████| 3127/3127 [00:00<00:00, 4592.68it/s]


In [8]:
# # UNTUK HAPUS FOLDER JIKA PERLU!

# import shutil

# # Specify the path to the non-empty directory
# directory_path = "/kaggle/working/wandb"

# try:
#     shutil.rmtree(directory_path)
#     print(f"Successfully deleted the entire directory tree: {directory_path}")
# except Exception as e:
#     print(f"Error while deleting: {e}")

In [9]:
# KAGGLE AUTOPILOT: OOD Splitter + Auto YAML (Bulletproof String Matching)
import requests
import os
import shutil
import time
import yaml
import re

# FUNGSI BARU: Normalisasi String
def normalize_name(name):
    # Menghapus semua karakter selain huruf dan angka, lalu ubah ke lowercase
    return re.sub(r'[^a-zA-Z0-9]', '', name).lower()

def get_exact_filenames_from_api(api_key, workspace, project, target_tag, ekspektasi):
    print(f"\n[>] Meminta daftar file untuk '{target_tag}' dari API...")
    search_url = f"https://api.roboflow.com/{workspace}/{project}/search?api_key={api_key}"
    
    attempt = 1
    max_attempts = 100 
    
    while attempt <= max_attempts:
        raw_data = []
        offset = 0
        
        while True:
            payload = {"limit": 250, "offset": offset, "fields": ["tags", "name"]}
            try:
                res = requests.post(search_url, json=payload, timeout=10)
                if res.status_code != 200:
                    print(f"    [!] Error API: {res.status_code}")
                    break
                    
                batch = res.json().get("results", [])
                if not batch: break
                raw_data.extend(batch)
                offset += 250
            except Exception as e:
                print(f"    [!] Koneksi API terputus: {e}")
                break

        unique_images = {img['id']: img for img in raw_data}.values()
        
        exact_names_normalized = []
        for img in unique_images:
            if target_tag in img.get("tags", []):
                nama_asli = img.get("name", "")
                nama_tanpa_ekstensi = os.path.splitext(nama_asli)[0]
                # NORMALISASI STRING API
                exact_names_normalized.append(normalize_name(nama_tanpa_ekstensi))
                
        total_ditemukan = len(exact_names_normalized)
        
        if total_ditemukan == ekspektasi:
            print(f"    [V] SUKSES! API menemukan {total_ditemukan} citra (Sesuai Ekspektasi).")
            return exact_names_normalized
        else:
            print(f"    [*] Percobaan {attempt}/{max_attempts}: Ditemukan {total_ditemukan}, Ekspektasi {ekspektasi}.")
            time.sleep(3)
            attempt += 1
            
    print(f"    [X] GAGAL: Mencapai batas maksimal percobaan untuk {target_tag}. Melewati tipe ini.")
    return None

def pisahkan_ood_hybrid(target_names_normalized, folder_asal, folder_tujuan):
    print(f"[>] Memindahkan file secara fisik ke {folder_tujuan.split('/')[-1]}...")
    os.makedirs(f"{folder_tujuan}/images", exist_ok=True)
    os.makedirs(f"{folder_tujuan}/labels", exist_ok=True)
    
    path_images = f"{folder_asal}/images"
    path_labels = f"{folder_asal}/labels"
    
    if not os.path.exists(path_images):
        print(f"    [!] FATAL: Folder asal '{path_images}' tidak ditemukan!")
        return

    pindah = 0
    for file_img in os.listdir(path_images):
        # 1. Buang bagian .rf. dan hash-nya
        base_name = file_img.split(".rf.")[0]
        
        # 2. Hapus injeksi ekstensi dari Roboflow (_png, _jpg, _jpeg) di akhir string
        # Regex ini mencari pola tersebut di akhir string ($)
        clean_local_name = re.sub(r'_(jpg|png|jpeg)$', '', base_name, flags=re.IGNORECASE)
        
        # 3. Normalisasi (hapus spasi, strip, dll)
        normalized_local = normalize_name(clean_local_name)
        
        # PENCOCOKAN YANG BENAR-BENAR KEBAL
        if normalized_local in target_names_normalized:
            src_img = os.path.join(path_images, file_img)
            dst_img = os.path.join(f"{folder_tujuan}/images", file_img)
            shutil.move(src_img, dst_img)
            
            file_txt = os.path.splitext(file_img)[0] + ".txt"
            src_lbl = os.path.join(path_labels, file_txt)
            dst_lbl = os.path.join(f"{folder_tujuan}/labels", file_txt)
            
            if os.path.exists(src_lbl):
                shutil.move(src_lbl, dst_lbl)
                pindah += 1
                
    print(f"    [V] SELESAI: Memindahkan {pindah} pasang file ke OOD.")

def generate_ood_yaml(base_yaml_path, ood_images_dir, output_yaml_path):
    print(f"[>] Mengekstraksi konfigurasi ke: {output_yaml_path.split('/')[-1]}...")
    if not os.path.exists(base_yaml_path):
        print(f"    [!] FATAL: File base YAML '{base_yaml_path}' tidak ditemukan.")
        return

    with open(base_yaml_path, 'r') as file:
        config = yaml.safe_load(file)

    config['test'] = os.path.abspath(ood_images_dir)

    with open(output_yaml_path, 'w') as file:
        yaml.dump(config, file, default_flow_style=False)
        
    print(f"    [V] YAML siap digunakan untuk model.val(data='{output_yaml_path.split('/')[-1]}')")

# ==========================================
# KONFIGURASI KAGGLE SAVE & COMMIT
# ==========================================
if __name__ == "__main__":
    API_KEY = "l71k5zH16yA3VFXEWrOH"
    WORKSPACE = "skripsian-bwueb"
    PROJECT = "skripsi_v1"
    
    # 1. PATH DATASET
    BASE_DATASET_PATH = "/kaggle/working/Skripsi_V1-4" 
    BASE_YAML_PATH = f"{BASE_DATASET_PATH}/data.yaml"
    # Karena Anda mengonfirmasi gambar ada di Test, kita kembalikan fokus ke folder Test saja
    FOLDER_ASAL = f"{BASE_DATASET_PATH}/test"
    
    # 2. DAFTAR TARGET OOD
    TARGET_CLASSES = {
        "type-009": 156,
        "type-021": 20,
        "type-011": 24
    }
    
    print("="*50)
    print(" MEMULAI AUTOPILOT PEMISAHAN OOD DATASET & YAML")
    print("="*50)
    
    for tag, ekspektasi in TARGET_CLASSES.items():
        FOLDER_TUJUAN = f"{BASE_DATASET_PATH}/test_ood_{tag}"
        OUTPUT_YAML = f"{BASE_DATASET_PATH}/test_ood_{tag}.yaml"
        
        daftar_nama_api = get_exact_filenames_from_api(API_KEY, WORKSPACE, PROJECT, tag, ekspektasi)
        
        if daftar_nama_api:
            pisahkan_ood_hybrid(daftar_nama_api, FOLDER_ASAL, FOLDER_TUJUAN)
            generate_ood_yaml(BASE_YAML_PATH, f"{FOLDER_TUJUAN}/images", OUTPUT_YAML)
            
    print("\n" + "="*50)
    print(" SELURUH PROSES PEMISAHAN SELESAI")
    print("="*50)

 MEMULAI AUTOPILOT PEMISAHAN OOD DATASET & YAML

[>] Meminta daftar file untuk 'type-009' dari API...
    [*] Percobaan 1/100: Ditemukan 130, Ekspektasi 156.
    [*] Percobaan 2/100: Ditemukan 110, Ekspektasi 156.
    [*] Percobaan 3/100: Ditemukan 141, Ekspektasi 156.
    [*] Percobaan 4/100: Ditemukan 140, Ekspektasi 156.
    [*] Percobaan 5/100: Ditemukan 137, Ekspektasi 156.
    [*] Percobaan 6/100: Ditemukan 149, Ekspektasi 156.
    [*] Percobaan 7/100: Ditemukan 141, Ekspektasi 156.
    [*] Percobaan 8/100: Ditemukan 138, Ekspektasi 156.
    [V] SUKSES! API menemukan 156 citra (Sesuai Ekspektasi).
[>] Memindahkan file secara fisik ke test_ood_type-009...
    [V] SELESAI: Memindahkan 156 pasang file ke OOD.
[>] Mengekstraksi konfigurasi ke: test_ood_type-009.yaml...
    [V] YAML siap digunakan untuk model.val(data='test_ood_type-009.yaml')

[>] Meminta daftar file untuk 'type-021' dari API...
    [*] Percobaan 1/100: Ditemukan 16, Ekspektasi 20.
    [*] Percobaan 2/100: Ditemukan 

In [10]:
# UNTUK CEK BOUNDING BOX TERKECIL YANG DIGUNAKAN UNTUK MENENTUKAN IMGSZ


# import os
# import pandas as pd

# def analyze_yolo_labels(label_dir, orig_w=1280, orig_h=720):
#     all_boxes = []
    
#     if not os.path.exists(label_dir):
#         print(f"[!] Path {label_dir} tidak ditemukan!")
#         return

#     for file in os.listdir(label_dir):
#         if file.endswith(".txt") and file != "classes.txt":
#             with open(os.path.join(label_dir, file), 'r') as f:
#                 for line in f.readlines():
#                     data = line.split()
#                     if len(data) == 5:
#                         # YOLO format: class x_center y_center width height
#                         class_id = int(data[0])
#                         w_norm = float(data[3])
#                         h_norm = float(data[4])
                        
#                         # Konversi ke pixel absolut
#                         pixel_w = w_norm * orig_w
#                         pixel_h = h_norm * orig_h
                        
#                         all_boxes.append({
#                             'file_name': file,
#                             'class_id': class_id,
#                             'w': round(pixel_w, 2), 
#                             'h': round(pixel_h, 2), 
#                             'area': round(pixel_w * pixel_h, 2)
#                         })

#     df = pd.DataFrame(all_boxes)
#     if df.empty:
#         print("[!] Tidak ada label ditemukan.")
#         return

#     print("=== STATISTIK BOUNDING BOX (DALAM PIXEL) ===")
#     print(f"Total Box Termuat: {len(df)}")
#     print(f"Lebar (W) - Min: {df['w'].min():.2f}, Max: {df['w'].max():.2f}, Mean: {df['w'].mean():.2f}")
#     print(f"Tinggi (H) - Min: {df['h'].min():.2f}, Max: {df['h'].max():.2f}, Mean: {df['h'].mean():.2f}")
#     print("-" * 60)
    
#     # Mencari 5 box terkecil untuk observasi
#     print("5 Box Terkecil (berdasarkan area untuk penentuan imgsz):")
    
#     # Memaksa pandas menampilkan tabel utuh tanpa terpotong
#     pd.set_option('display.max_columns', None)
#     pd.set_option('display.width', 1000)
#     print(df.nsmallest(5, 'area').to_string(index=False))

# # ==========================================
# # CARA PENGGUNAAN DI KAGGLE
# # ==========================================
# # Pastikan path mengarah ke folder dataset Anda yang belum di-resize
# PATH_LABELS = "/kaggle/working/Skripsi_V1-4/train/labels"
# analyze_yolo_labels(PATH_LABELS)

## **STEP 3: Start Custom Training!**

### 3.1 - SETUP WANDB FOR MONITORING

In [11]:
# Set integrasi YOLO dan W&B
!yolo settings wandb=true

# Login ke W&B
wandb.login(key=WANDB_API_KEY)

FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
JSONDict("/root/.config/yolov12/settings.json"):
{
  "settings_version": "0.0.6",
  "datasets_dir": "/kaggle/working/datasets",
  "weights_dir": "weights",
  "runs_dir": "runs",
  "uuid": "1bfc3e992d24318da58ddee183be5bf9388a31f26bab1738e986ec4d297417ff",
  "sync": true,
  "api_key": "",
  "openai_api_key": "",
  "clearml": true,
  "comet": true,
  "dvc": true,
  "hub": true,
  "mlflow": true,
  "neptune": true,
  "raytune": true,
  "tensorboard": true,
  "wandb": true,
  "vscode_msg": true
}
💡 Learn more about Ultralytics Settings at https://docs.ultralytics.com/quickstart/#ultralytics-settings


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aizarhafizh (aizarhafizh-universitas-gadjah-mada-library) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### 3.2 Train <span style="color:yellow">YOLOv12-nano</span>

In [12]:
# from ultralytics import YOLO
# os.chdir("/kaggle/working")

# # Inisialisasi proyek di W&B
# # wandb.init(project="skripsi-YOLOv12", job_type="training")

# # Ubah inisialisasi bobot sesuai sesi eksperimen:
# # yolov12n.pt, yolov12s.pt, yolov12m.pt, yolov12l.pt, atau yolov12x.pt
# varian_model = 'yolov12n.pt' 
# model = YOLO(varian_model)

# results = model.train(
#     # Pastikan path ini mengarah ke dataset Versi 3 (resolusi asli, tanpa resize Roboflow)
#     data='/kaggle/working/Skripsi_V1-4/data.yaml', 
#     project='skripsi_yolov12',
#     name=f'train_{varian_model.split(".")[0]}_1024px',
    
#     # --- PARAMETER ARSITEKTUR & KOMPUTASI ---
#     epochs=300,          # Set tinggi, kita akan mengandalkan Early Stopping
#     patience=50,         # Beri ruang 50 epoch tanpa peningkatan sebelum dihentikan paksa
#     imgsz=1024,          # Resolusi Sweet Spot (menjaga kotak 16px tetap di atas batas P3 stride)
#     batch=16,            # !!! UBAH NILAI INI BERDASARKAN VARIAN MODEL (Lihat panduan di bawah) !!!
#     device=[0,1],            # Paksa jalan di GPU
#     optimizer='auto',    # Biarkan algoritma memilih AdamW atau SGD berdasarkan ukuran model
    
#     # --- AUGMENTASI ANTI-DISTORSI OCR YANG SUDAH DIREVISI ---
#     fliplr=0.0,          
#     flipud=0.0,          
#     degrees=5.0,         
#     hsv_h=0.015,         
#     hsv_s=0.5,           
#     hsv_v=0.4,           
#     mosaic=1.0,          
#     close_mosaic=10,
    
#     # --- PENGUNCI KEAMANAN OCR BARU ---
#     erasing=0.0,         # FATAL: Matikan Random Erasing agar angka tidak tertutup kotak hitam
#     auto_augment=False,  # FATAL: Matikan RandAugment agar tidak ada distorsi warna/bentuk liar
#     scale=0.2            # Kurangi efek zoom agar objek teks 16px tidak ciut menjadi 8px
# )

# wandb.finish()

## 3.2.1 TRAIN MODEL

In [13]:
import wandb
import os
import glob
import gc
import time
import torch
from ultralytics import YOLO

MODEL_NAME = 'yolov12s.pt'  # Ganti dengan 'yolov12l.pt' atau 'yolov12x.pt' sesuai sesi
IMG_SIZE = 1024             # Fiksasi pada 1024px
BATCH_SIZE = 16             # Sesuaikan dengan tabel referensi di atas
WORKERS = 4

PATH_DATASET = "/kaggle/working/Skripsi_V1-4"
PROJECT_NAME = "yolov12_final"

# Ekstraksi nama varian (misal: 'yolov12n', 'yolov12x') untuk penamaan dinamis
variant_prefix = MODEL_NAME.split('.')[0]
run_name = f"{variant_prefix}_aug_{IMG_SIZE}px_final"

# =========================================================
# 2. DEFINISI AUGMENTASI & PATH OOD
# =========================================================
custom_aug_params = {
    'fliplr': 0.0, 'flipud': 0.0, 'erasing': 0.0, 'auto_augment': False,
    'shear': 0.0, 'perspective': 0.0, 'mixup': 0.0, 'copy_paste': 0.0,
    'scale': 0.2, 'translate': 0.2, 'degrees': 5.0,
    'hsv_h': 0.015, 'hsv_s': 0.5, 'hsv_v': 0.4,
    'mosaic': 1.0, 'close_mosaic': 10
}

test_yamls = {
    "Reguler": f"{PATH_DATASET}/data.yaml",
    "009_Kiri": f"{PATH_DATASET}/test_ood_type-009.yaml",
    "021_Grid": f"{PATH_DATASET}/test_ood_type-021.yaml",
    "011_Kanan": f"{PATH_DATASET}/test_ood_type-011.yaml"
}

print(f"=== MEMULAI SINGLE RUN: {run_name} (2 GPU DDP) ===")

if wandb.run is not None:
    wandb.finish()
    time.sleep(5)

# =========================================================
# FASE 1: TRAINING DDP (2 GPU)
# =========================================================
model = YOLO(MODEL_NAME)

train_args = {
    'data': f'{PATH_DATASET}/data.yaml',
    'project': PROJECT_NAME,
    'name': f"train_{run_name}",
    'epochs': 300,
    'patience': 50,
    'imgsz': IMG_SIZE,
    'batch': BATCH_SIZE,  
    'device': [0, 1],
    'workers': WORKERS,
    'optimizer': 'auto',
    'exist_ok': True
}

train_args.update(custom_aug_params)

# Ultralytics akan meluncurkan Subprocess DDP secara otomatis
model.train(**train_args)

if wandb.run is not None:
    wandb.finish()
    time.sleep(5)

# Pembersihan VRAM sebelum masuk ke Fase Evaluasi
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
time.sleep(5)

# =========================================================
# EVALUASI OOD & LOGGING ARTIFACT
# =========================================================
print(f"\n[{run_name}] MEMULAI EVALUASI MODEL...")

save_dir = os.path.join("/kaggle/working", PROJECT_NAME, f"train_{run_name}")
best_weight_path = os.path.join(save_dir, 'weights', 'best.pt')

if not os.path.exists(best_weight_path):
    print(f"[{run_name}] FATAL ERROR: best.pt tidak ditemukan di {best_weight_path}")
    exit()

# Buka run W&B baru khusus untuk merekam metrik evaluasi
eval_run = wandb.init(
    project=PROJECT_NAME,
    name=f"eval_{run_name}",
    job_type="evaluation",
    config=train_args,
    settings=wandb.Settings(reinit=True)
)

# Upload Model Artifact secara manual agar penamaan bersih
artifact = wandb.Artifact(
    name=run_name,
    type="model",
    description=f"Best weights for {run_name}"
)
artifact.add_file(best_weight_path)
eval_run.log_artifact(artifact, aliases=["best_model_candidate"])
print(f"[{run_name}] Bobot berhasil diunggah ke W&B Artifacts.")

# Load Model Terbaik untuk Pengujian
best_model = YOLO(best_weight_path)
test_metrics_for_wandb = {}

# Eksekusi Loop Pengujian ke 4 Dataset
for nama_tes, path_yaml in test_yamls.items():
    eval_name = f"predict_{run_name}_{nama_tes}"
    
    metrics = best_model.val(
        project=PROJECT_NAME,
        data=path_yaml,
        split='test',
        imgsz=IMG_SIZE,
        save=True,
        name=eval_name
    )
    
    # Ekstraksi mAP50-95
    test_metrics_for_wandb[f"Test_mAP50-95/{nama_tes}"] = metrics.box.map * 100
    
    # Ekstraksi Gambar Prediksi (Ambil 3 gambar pertama)
    pred_dir = f"/kaggle/working/{PROJECT_NAME}/{eval_name}"
    pred_images_paths = glob.glob(os.path.join(pred_dir, "*_pred.jpg"))
    if pred_images_paths:
        test_metrics_for_wandb[f"Predictions/{nama_tes}"] = [wandb.Image(img) for img in pred_images_paths[:3]]

# Kirim dan Tutup W&B
eval_run.log(test_metrics_for_wandb)
eval_run.finish()
print(f"[{run_name}] Evaluasi selesai dan metrik dikirim ke W&B.")

print(f"\n=== SINGLE RUN {run_name} SELESAI ===")

=== MEMULAI SINGLE RUN: yolov12s_aug_1024px_final (2 GPU DDP) ===


100%|██████████| 17.8M/17.8M [00:00<00:00, 30.4MB/s]


New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.63 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                        CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=yolov12s.pt, data=/kaggle/working/Skripsi_V1-4/data.yaml, epochs=300, time=None, patience=50, batch=16, imgsz=1024, save=True, save_period=-1, cache=False, device=[0, 1], workers=4, project=yolov12_final, name=train_yolov12s_aug_1024px_final, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_b

100%|██████████| 755k/755k [00:00<00:00, 47.6MB/s]
E0000 00:00:1777225027.111495      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777225027.208945      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777225028.113622      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777225028.113676      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777225028.113679      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777225028.113682      24 computation_placer.cc:177] comput

Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1      9344  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2, 1, 2]          
  2                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  3                  -1  1     37120  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2, 1, 4]        
  4                  -1  1    103360  ultralytics.nn.modules.block.C3k2            [128, 256, 1, False, 0.25]    
  5                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  6                  -1  2    677120  ultralytics.nn.modules.block.A2C2f           [256, 256, 2, True, 4]        
  7                  -1  1   1180672  ultralytics

...

val: New cache created: /kaggle/working/Skripsi_V1-4/test_ood_type-011/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:01<00:00,  1.89it/s]


                   all         24        130      0.971      0.991      0.991       0.95
                BP_DIA         23         24      0.962      0.958      0.972      0.938
               BP_MEAN         22         22          1      0.986      0.995      0.843
                BP_SYS         23         23      0.969          1      0.995      0.965
                    HR         24         24       0.97          1      0.995      0.978
                    RR         19         19      0.963          1      0.995      0.983
                  SPO2         18         18      0.961          1      0.995      0.995
Speed: 0.4ms preprocess, 28.1ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to yolov12_final/predict_yolov12s_aug_1024px_final_011_Kanan


wandb: updating run metadata
wandb: uploading media/images/Predictions/Reguler_0_3c6004c786b2d8b57c45.jpg; uploading media/images/Predictions/Reguler_0_c5faef95284d1e0bf4b6.jpg; uploading media/images/Predictions/009_Kiri_0_e873ef2b320f7677a3da.jpg; uploading media/images/Predictions/009_Kiri_0_72ae4137fd23a7a77498.jpg; uploading media/images/Predictions/021_Grid_0_1433667a63f29392ca97.jpg (+ 8 more)
wandb: uploading media/images/Predictions/Reguler_0_c5faef95284d1e0bf4b6.jpg; uploading media/images/Predictions/009_Kiri_0_72ae4137fd23a7a77498.jpg
wandb: 
wandb: Run history:
wandb:  Test_mAP50-95/009_Kiri ▁
wandb: Test_mAP50-95/011_Kanan ▁
wandb:  Test_mAP50-95/021_Grid ▁
wandb:   Test_mAP50-95/Reguler ▁
wandb: 
wandb: Run summary:
wandb:  Test_mAP50-95/009_Kiri 76.22773
wandb: Test_mAP50-95/011_Kanan 95.02641
wandb:  Test_mAP50-95/021_Grid 69.26082
wandb:   Test_mAP50-95/Reguler 96.28096
wandb: 
wandb: 🚀 View run eval_yolov12s_aug_1024px_final at: https://wandb.ai/aizarhafizh-universit

[yolov12s_aug_1024px_final] Evaluasi selesai dan metrik dikirim ke W&B.

=== SINGLE RUN yolov12s_aug_1024px_final SELESAI ===


### 3.3 - TEST

In [14]:
# import wandb
# import glob
# import os
# from ultralytics import YOLO

# # Inisialisasi W&B
# wandb.init(project="skripsi_yolov12", name="eval_yolov12n_no_aug_640px")

# # Muat bobot model
# best_model = YOLO('/kaggle/input/models/aizarhafizhsugm/yolo12n-base-specific-ultralytics/pytorch/default/1/best.pt')

# test_yamls = {
#     "reguler": "/kaggle/working/Skripsi_V1-4/data.yaml",
#     "type009_Kiri": "/kaggle/working/Skripsi_V1-4/test_ood_type-009.yaml",
#     "type021_Grid": "/kaggle/working/Skripsi_V1-4/test_ood_type-021.yaml",
#     "type011_Kanan": "/kaggle/working/Skripsi_V1-4/test_ood_type-011.yaml"
# }

# # Dictionary untuk menampung metrik yang akan dikirim ke W&B
# test_metrics_for_wandb = {}

# print("=== MEMULAI VALIDASI TEST SET BATCH ===")

# # Eksekusi Loop Evaluasi
# for nama_tes, path_yaml in test_yamls.items():
#     print(f"\n[>] Memproses Test Set: {nama_tes}")
    
#     # Nama sub-folder prediksi spesifik untuk test set ini
#     eval_name = f"eval_yolov12n_1024px_{nama_tes}"
    
#     # Jalankan evaluasi YOLO
#     metrics = best_model.val(
#         project='skripsi_yolov12',
#         data=path_yaml, 
#         split='test', 
#         imgsz=640,
#         save=True, 
#         name=eval_name 
#     )
    
#     # Ekstrak nilai mAP (dikali 100 agar menjadi persentase)
#     map50 = metrics.box.map50 * 100
#     map50_95 = metrics.box.map * 100
    
#     print(f"    [V] HASIL KAGGLE -> mAP@50: {map50:.2f}%, mAP@50-95: {map50_95:.2f}%")
    
#     # Simpan metrik angka ke dictionary
#     test_metrics_for_wandb[f"Test_mAP50/{nama_tes}"] = map50
#     test_metrics_for_wandb[f"Test_mAP50-95/{nama_tes}"] = map50_95

#     # --- BAGIAN BARU: MENGAMBIL GAMBAR PREDIKSI ---
#     # YOLO menyimpan hasil gambar dengan akhiran _pred.jpg di folder project/name
#     save_dir = f"/kaggle/working/skripsi_yolov12/{eval_name}"
#     pred_images_paths = glob.glob(os.path.join(save_dir, "*_pred.jpg"))
    
#     if pred_images_paths:
#         # Opsional: Batasi upload ke 3 gambar (batch) pertama agar log W&B tidak terlalu berat
#         # Anda bisa menghapus [:3] jika ingin mengunggah seluruh batch gambar
#         wandb_images = [wandb.Image(img_path) for img_path in pred_images_paths[:3]]
        
#         # Kirim objek gambar tersebut ke W&B di bawah panel "Predictions"
#         test_metrics_for_wandb[f"Predictions/{nama_tes}"] = wandb_images
#         print(f"    [+] Berhasil membungkus {len(wandb_images)} gambar prediksi ke W&B.")

# # Tembakkan semua hasil ke W&B secara serentak
# wandb.log(test_metrics_for_wandb)
# wandb.finish()

# print("\n=== EVALUASI SELESAI. METRIK & GAMBAR TELAH DIKIRIM KE W&B ===")

In [15]:
# # CEK ISI YAML FILE

# import os
# # 1. Get file descriptor
# fd = os.open("/kaggle/working/Skripsi_V1-4/test_ood_type-009.yaml", os.O_RDONLY)

# # 2. Read up to 1024 bytes
# data = os.read(fd, 1024)
# print(data.decode('utf-8'))

# # 3. Close the descriptor
# os.close(fd)